# Nsquare | YOLO 객체 탐지 실습 — 2단계: **모델 구조까지 자유롭게 설계**

**목표:** 1단계와 **완전히 동일한 데이터 및 평가 조건**에서 Backbone뿐 아니라 Detection Head, Custom CNN 등 **모델 아키텍처 자체**까지 변경하여 Validation mAP@0.5를 개선합니다. 각 수강생은 RTX 5080 한 장을 사용한다고 가정합니다.

> **수정 가능한 것:** ResNet/MobileNet/EfficientNet/직접 작성한 PyTorch 아키텍처, Head 채널 수, 학습 Hyperparameter 및 전략. **공통 제약:** 동일한 Train/Val/Test 분할, 입력 224×224, YOLO-style 예측 텐서 `(N,7,7,16)` 인터페이스 및 VOC 11-point mAP@0.5 평가. 모델의 내부 구조는 제한하지 않습니다.

**진행:** 1단계와 같은 Baseline 실행 → 다른 모델 또는 `StudentCNN`을 시도 → 필요하면 직접 `nn.Module`을 등록 → Validation 기준 선택 → 최종 Test 1회.


## 1. 환경 및 동일 데이터 준비

| 항목 | 1단계 / 2단계 공통 값 |
|---|---|
| 데이터 | PASCAL VOC 2012, 6개 클래스 (`bicycle`, `bus`, `car`, `cat`, `dog`, `person`) |
| Train / Validation | VOC 공식 `train.txt`에서 seed 42로 80:20 분리 후 각 **1,200장 / 300장** |
| Final Test | VOC 공식 `val.txt` 전체 (실험 중 사용 금지) |
| 이미지 | **224×224**, ImageNet 정규화 |
| Grid / Box | `7×7`, 셀당 2 boxes, 출력 16 채널 |
| Baseline | **ResNet18, ImageNet 사전학습, AdamW, lr=3e-4, batch=16, 4 epochs** |
| Metric | VOC 11-point **mAP@0.5** (`conf=0.01`, `NMS IoU=0.5`) |
| 기본 Validation 주기 | 2 Epoch마다 + 마지막 Epoch |

**주의:** 첫 데이터 다운로드/압축 해제 및 Backbone 가중치 다운로드는 네트워크 환경에 따라 오래 걸릴 수 있습니다. 두 노트북 모두 원본의 VOC2012 데이터 다운로드 URL을 그대로 사용합니다.


In [ ]:
# Colab에서 기본 제공되지 않은 패키지만 설치합니다.
import importlib.util
import subprocess
import sys

need = {'cv2': 'opencv-python-headless', 'matplotlib': 'matplotlib', 'tqdm': 'tqdm', 'pandas': 'pandas'}
missing = [package for module, package in need.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import os, random, shutil, zipfile, copy, gc, time
from pathlib import Path
import xml.etree.ElementTree as ET

import cv2
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.datasets import VOCDetection

SEED = 42

def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

reset_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('Device :', device, '| CUDA GPU:', torch.cuda.get_device_name(0) if device.type == 'cuda' else '없음 (GPU 사용 권장)')

In [ ]:
# 제공된 원본 데이터 URL 및 주석 파일을 그대로 사용합니다.
DATA_URL = 'https://o365uos-my.sharepoint.com/:u:/g/personal/hjseong_office_uos_ac_kr/EQPy0vqC_-5Lj7G6Z_tslc0BsjvqgZ0LGymvuix3T4WGeQ?e=hqy88B&download=1'
DATA_DIR = Path('./dataset')
ARCHIVE = DATA_DIR / 'VOC2012_train_val.zip'
VOC_DIR = DATA_DIR / 'VOCdevkit' / 'VOC2012'
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (VOC_DIR / 'ImageSets' / 'Main' / 'train.txt').is_file():
    if not ARCHIVE.is_file() or not zipfile.is_zipfile(ARCHIVE):
        print('원본 VOC2012 데이터 다운로드 시도...')
        try:
            subprocess.run(['wget', '-q', '--show-progress', '--tries=3', '--timeout=30',
                            DATA_URL, '-O', str(ARCHIVE)], check=True)
            if not zipfile.is_zipfile(ARCHIVE):
                raise RuntimeError('다운로드된 파일이 유효한 ZIP이 아닙니다.')
        except Exception as e:
            if ARCHIVE.exists():
                ARCHIVE.unlink()
            print('자동 다운로드 실패:', e)
            print('원본 VOC2012_train_val.zip 파일을 직접 업로드해 주세요.')
            try:
                from google.colab import files
                uploaded = files.upload()
                if 'VOC2012_train_val.zip' not in uploaded:
                    raise FileNotFoundError('원본 VOC2012_train_val.zip 파일이 필요합니다.')
                ARCHIVE.write_bytes(uploaded['VOC2012_train_val.zip'])
            except ImportError as exc:
                raise RuntimeError(f'제공된 원본 ZIP을 {ARCHIVE} 위치에 넣은 뒤 재실행하세요.') from exc
    assert zipfile.is_zipfile(ARCHIVE), '유효한 원본 VOC ZIP 파일이 필요합니다.'
    print('압축 해제 중...')
    with zipfile.ZipFile(ARCHIVE, 'r') as zf:
        zf.extractall(DATA_DIR)

    # 제공된 압축파일의 폴더 구조가 다르더라도 동일 VOC2012 디렉터리로 배치합니다.
    if not (VOC_DIR / 'ImageSets' / 'Main' / 'train.txt').is_file():
        candidates = [p.parent for p in DATA_DIR.rglob('JPEGImages')
                      if p.is_dir() and (p.parent / 'Annotations').is_dir()
                      and (p.parent / 'ImageSets' / 'Main' / 'train.txt').is_file()]
        if not candidates:
            raise FileNotFoundError('압축파일에서 VOC2012/JPEGImages, Annotations, ImageSets/Main/train.txt를 찾을 수 없습니다.')
        src_dir = candidates[0]
        if src_dir.resolve() != VOC_DIR.resolve():
            VOC_DIR.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src_dir), str(VOC_DIR))

assert (VOC_DIR / 'ImageSets' / 'Main' / 'train.txt').is_file(), 'VOC2012 train.txt를 찾을 수 없습니다.'
assert (VOC_DIR / 'ImageSets' / 'Main' / 'val.txt').is_file(), 'VOC2012 val.txt를 찾을 수 없습니다.'
DATA_PATH = str(DATA_DIR)
print('데이터 준비 완료:', VOC_DIR)

### 데이터 전처리와 Target 변환 (공통 코드)

6개 클래스 필터링, VOC annotation 읽기, YOLO Target 인코딩 `(7,7,11)`, 고정 데이터 분할은 **동일하게 유지**합니다. Train에만 증강을 적용하고 Validation/Test에는 동일한 고정 변환을 사용합니다. 두 단계의 공정한 비교를 위해 다음 셀의 `IMAGE_SIZE`, `TRAIN_SAMPLES`, `VAL_SAMPLES`, `SEED`, 평가 함수는 변경하지 않습니다.


In [ ]:
# YOLOv1-style Target encoding (6 classes / 7x7 grid) / resolution-aware augmentation.

import xml.etree.ElementTree as ET

# We use 6 classes in PASCAL VOC dataset
VOC_CLASSES = (
    'bicycle', 'bus', 'car', 'cat', 'dog', 'person'
    # 'aeroplane', 'bird', 'boat', 'bottle',
    # 'chair', 'cow', 'diningtable', 'horse',
    # 'motorbike', 'pottedplant',
    # 'sheep', 'sofa', 'train', 'tvmonitor'
)

CLASS_TO_IDX = {name: idx for idx, name in enumerate(VOC_CLASSES)}
NUM_CLASSES = len(VOC_CLASSES) # Should be 6
TARGET_CLASSES_SET = set(VOC_CLASSES)

# This class wraps VOCDetection to filter out images (20 classes -> 6 classes)
# that do NOT contain any of the classes in `target_classes`
class FilteredVOCDetection(VOCDetection):
    def __init__(self, root, year, image_set, download, target_classes: set, transforms=None):

        # Initialize the parent class, but WITHOUT transforms
        # We will apply transforms manually in __getitem__
        super().__init__(root=root, year=year, image_set=image_set, download=download, transforms=None)

        self.target_classes = target_classes
        # Store the joint transform
        self.joint_transform = transforms

        print(f"Filtering dataset '{image_set}' for {len(self.target_classes)} target classes...")
        self._filter_dataset()
        # self.images is overwritten, so len(self.images) will be correct
        print(f"Filtering complete. Found {len(self.images)} valid images out of original count.")

    def _filter_dataset(self):
        # self.annotations is a property, self.images is a list
        valid_indices = []
        original_image_paths = self.images
        original_annot_paths = self.annotations # Read from the property

        for i in tqdm(range(len(original_annot_paths)), desc="Filtering Images"):
            annot_path = original_annot_paths[i]

            try:
                tree = ET.parse(annot_path)
                root = tree.getroot()
                found = False

                for obj in root.findall('object'):
                    class_name = obj.find('name').text
                    if class_name in self.target_classes:
                        found = True
                        break # Found a valid object, no need to check others

                if found:
                    valid_indices.append(i)
            except Exception as e:
                img_name = os.path.basename(original_image_paths[i])
                print(f"Warning: Could not parse {annot_path} (image: {img_name}). Error: {e}. Skipping image.")

        # Overwrite the parent class's image list
        # This works because self.images is a simple list attribute
        self.images = [original_image_paths[i] for i in valid_indices]

        # Store the filtered list in a new attribute
        self.filtered_annotations = [original_annot_paths[i] for i in valid_indices]

    def __getitem__(self, index):
        # This implementation is copied from the parent VOCDetection.__getitem__
        # to ensure we load the raw data

        # self.images was successfully overwritten, so this is correct
        img = Image.open(self.images[index]).convert('RGB')

        # Read from filtered_annotations list instead of the parent's property
        target = self.parse_voc_xml(ET.parse(self.filtered_annotations[index]).getroot())

        # Now, apply the joint transform that we stored
        if self.joint_transform:
            img, target = self.joint_transform(img, target)

        return img, target

# Converts VOCDetection's XML target to YOLO's (S, S, C+5) tensor.
class EncodeYoloTarget:
    def __init__(self, S=7, C=6):
        self.S = S
        self.C = C
        self.target_tensor_shape = (S, S, C + 5) # (x, y, w, h, conf) + C_classes

    def __call__(self, target):
        # target: annotation dictionary returned by VOCDetection
        target_tensor = torch.zeros(self.target_tensor_shape)

        try:
            img_w = int(target['annotation']['size']['width'])
            img_h = int(target['annotation']['size']['height'])

            # Objects are returned as a list if multiple, or a dict if single
            try:
                objs = target['annotation']['object']
                if not isinstance(objs, list):
                    objs = [objs]
            except KeyError:
                objs = [] # Handle images with no objects

            for obj in objs:
                class_name = obj['name']

                # This check uses the global CLASS_TO_IDX
                if class_name not in CLASS_TO_IDX:
                    continue # Skip objects that are not in our 6 target classes

                class_idx = CLASS_TO_IDX[class_name]

                bbox = obj['bndbox']
                # Use float for more precision before int casting
                xmin = float(bbox['xmin'])
                ymin = float(bbox['ymin'])
                xmax = float(bbox['xmax'])
                ymax = float(bbox['ymax'])

                # Skip invalid boxes (e.g., after augmentation)
                if xmax <= xmin or ymax <= ymin:
                    continue

                # Convert to YOLO format (x_center, y_center, w, h) - relative to image (0~1)
                x_center = (xmin + xmax) / 2 / img_w
                y_center = (ymin + ymax) / 2 / img_h
                w = (xmax - xmin) / img_w
                h = (ymax - ymin) / img_h

                # Calculate cell index
                grid_x_idx = int(x_center * self.S)
                grid_y_idx = int(y_center * self.S)

                # Clamp values to be within grid [0, S-1]
                # This handles edge cases where x_center or y_center is exactly 1.0
                grid_x_idx = min(grid_x_idx, self.S - 1)
                grid_y_idx = min(grid_y_idx, self.S - 1)

                # Coordinates within the cell (0~1)
                x_cell = (x_center * self.S) - grid_x_idx
                y_cell = (y_center * self.S) - grid_y_idx

                # Check if this cell is already responsible for an object (YOLOv1: one object per cell)
                if target_tensor[grid_y_idx, grid_x_idx, self.C] == 0:
                    # Confidence
                    target_tensor[grid_y_idx, grid_x_idx, self.C] = 1
                    # Bbox (x_cell, y_cell, w_img, h_img)
                    target_tensor[grid_y_idx, grid_x_idx, (self.C + 1):(self.C + 5)] = torch.tensor([x_cell, y_cell, w, h])
                    # Class
                    target_tensor[grid_y_idx, grid_x_idx, class_idx] = 1

        except Exception as e:
            # Print the specific target that failed, which is more useful
            img_filename = target.get('annotation', {}).get('filename', 'Unknown')
            print(f"Warning: Error parsing target for {img_filename}. Error: {e}")
            # Return an empty tensor
            return torch.zeros(self.target_tensor_shape)

        return target_tensor

# Handles augmentations that must be applied to both image and target (bboxes)
class JointTransform:
    def __init__(self, S=7, C=6, input_size=(448, 448), augment=True,
                 scale_range_aug: tuple = (0.7, 1.3)):
        self.S = S
        self.C = C
        self.input_size = input_size
        self.augment = augment

        # The range for random scaling (e.g., 0.7x to 1.3x)
        self.scale_range_aug = scale_range_aug
        # The fill color for empty areas after translation (128, 128, 128) is gray
        self.padding_fill = (128, 128, 128)

        # This is the target encoder
        self.target_encoder = EncodeYoloTarget(S=S, C=C)

        # Augmentation transforms (image only)
        self.color_jitter = transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1)

        # Final image transforms (applied after augmentations)
        self.final_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __call__(self, img_pil: Image.Image, target_dict: dict):
        # img_pil: PIL.Image
        # target_dict: annotation dictionary

        # Get original image size
        original_w = int(target_dict['annotation']['size']['width'])
        original_h = int(target_dict['annotation']['size']['height'])

        # Get original boxes
        # It's easier to work with a simple list of [xmin, ymin, xmax, ymax, name]
        original_boxes = []
        try:
            objs = target_dict['annotation']['object']
            if not isinstance(objs, list):
                objs = [objs] # Handle single object case
            for obj in objs:
                bbox = obj['bndbox']
                original_boxes.append([
                    int(float(bbox['xmin'])), int(float(bbox['ymin'])),
                    int(float(bbox['xmax'])), int(float(bbox['ymax'])),
                    obj['name']
                ])
        except KeyError:
            pass # No objects in this image

        # Apply Augmentations (if enabled)
        if self.augment:
            # Color Jitter (Image only)
            img_pil = self.color_jitter(img_pil)

            # Random Scaling and Translation (Jitter)
            # This logic replaces the simple deterministic resize

            # Get a random scale factor
            scale = random.uniform(self.scale_range_aug[0], self.scale_range_aug[1])
            # Scale around the MODEL INPUT size, not the original VOC image size.
            # This ensures that 224x224 inputs preserve objects as intended.
            canvas_w, canvas_h = self.input_size
            scaled_w = max(1, int(round(canvas_w * scale)))
            scaled_h = max(1, int(round(canvas_h * scale)))
            sx = scaled_w / original_w
            sy = scaled_h / original_h

            # Resize the image
            img_pil = img_pil.resize((scaled_w, scaled_h), Image.BILINEAR)

            # Create a canvas of model-input size (224x224 by default).
            canvas = Image.new('RGB', self.input_size, self.padding_fill)

            # Calculate random translation (dx, dy)
            # This determines where the top-left corner of the scaled image is placed
            dx = int(random.uniform(min(0, canvas_w - scaled_w), max(0, canvas_w - scaled_w)))
            dy = int(random.uniform(min(0, canvas_h - scaled_h), max(0, canvas_h - scaled_h)))

            # Paste the scaled image onto the canvas
            canvas.paste(img_pil, (dx, dy))
            img_pil = canvas # This canvas is now our augmented image

            # Update boxes for BOTH scaling and translation
            scaled_translated_boxes = []
            for (xmin, ymin, xmax, ymax, class_name) in original_boxes:
                # Apply scaling
                xmin_s, ymin_s = xmin * sx, ymin * sy
                xmax_s, ymax_s = xmax * sx, ymax * sy

                # Apply translation
                xmin_t, ymin_t = xmin_s + dx, ymin_s + dy
                xmax_t, ymax_t = xmax_s + dx, ymax_s + dy

                # Clip boxes to the canvas boundaries [0, 0, canvas_w, canvas_h]
                xmin_c = max(0, xmin_t)
                ymin_c = max(0, ymin_t)
                xmax_c = min(canvas_w, xmax_t)
                ymax_c = min(canvas_h, ymax_t)

                # Filter boxes that have zero area after clipping
                if xmax_c > xmin_c and ymax_c > ymin_c:
                    scaled_translated_boxes.append([xmin_c, ymin_c, xmax_c, ymax_c, class_name])

            # Random Horizontal Flip
            final_boxes = []
            if random.random() < 0.5:
                img_pil = TF.hflip(img_pil)

                for (xmin, ymin, xmax, ymax, class_name) in scaled_translated_boxes:
                    # Flipped xmin is (canvas_width - original xmax)
                    # Flipped xmax is (canvas_width - original xmin)
                    flipped_xmin = canvas_w - xmax
                    flipped_xmax = canvas_w - xmin
                    final_boxes.append([flipped_xmin, ymin, flipped_xmax, ymax, class_name])
            else:
                final_boxes = scaled_translated_boxes

            # Rebuild target_dict
            # The target_dict must now reflect the new model input size
            target_dict['annotation']['size']['width'] = str(canvas_w)
            target_dict['annotation']['size']['height'] = str(canvas_h)

            new_obj_list = []
            for (xmin, ymin, xmax, ymax, class_name) in final_boxes:
                new_obj_list.append({
                    'name': class_name,
                    'bndbox': {
                        'xmin': str(int(xmin)),
                        'ymin': str(int(ymin)),
                        'xmax': str(int(xmax)),
                        'ymax': str(int(ymax))
                    }
                })
            target_dict['annotation']['object'] = new_obj_list

        # Deterministic Resize (for Validation/Test)
        else:
            # If not augmenting, just do the original deterministic resize
            img_pil = TF.resize(img_pil, self.input_size)
            new_w, new_h = self.input_size

            # Calculate scale factors
            w_scale = new_w / original_w
            h_scale = new_h / original_h

            # Update target size AND scale bounding boxes
            target_dict['annotation']['size']['width'] = str(new_w)
            target_dict['annotation']['size']['height'] = str(new_h)

            # We must rebuild the dictionary here too, to use the original_boxes
            new_obj_list = []
            for (xmin, ymin, xmax, ymax, class_name) in original_boxes:
                # Scale coordinates and clamp to new boundaries
                xmin_s = max(0, min(new_w, int(xmin * w_scale)))
                ymin_s = max(0, min(new_h, int(ymin * h_scale)))
                xmax_s = max(0, min(new_w, int(xmax * w_scale)))
                ymax_s = max(0, min(new_h, int(ymax * h_scale)))

                if xmax_s > xmin_s and ymax_s > ymin_s:
                    new_obj_list.append({
                        'name': class_name,
                        'bndbox': {
                            'xmin': str(xmin_s),
                            'ymin': str(ymin_s),
                            'xmax': str(xmax_s),
                            'ymax': str(ymax_s)
                        }
                    })
            target_dict['annotation']['object'] = new_obj_list


        # Final Image Transform (to Tensor)
        img_tensor = self.final_transform(img_pil)

        # Target Transform (to Tensor)
        # Now, target_encoder receives coordinates relative to the resized model input image
        # It will also filter out non-target classes
        target_tensor = self.target_encoder(target_dict)

        return img_tensor, target_tensor

In [ ]:
# 고정 데이터 subset: 모든 실험에서 그대로 사용하고 Test는 절대 샘플링/학습에 포함하지 않습니다.
DATA_PATH = './dataset'
IMAGE_SIZE = 224                       # 필요하면 256/320으로 변경 가능 (모델은 adaptive pool 사용)
INPUT_SIZE = (IMAGE_SIZE, IMAGE_SIZE)
TRAIN_SAMPLES = 1200                    # 더 빠른 체험: 600 / 권장: 1200
VAL_SAMPLES = 300                       # 더 빠른 체험: 150 / 권장: 300
S, B, C = 7, 2, len(VOC_CLASSES)

train_transform = JointTransform(S=S, C=C, input_size=INPUT_SIZE, augment=True)
validation_transform = JointTransform(S=S, C=C, input_size=INPUT_SIZE, augment=False)

train_full_aug = FilteredVOCDetection(
    root=DATA_PATH, year='2012', image_set='train', download=False,
    target_classes=TARGET_CLASSES_SET, transforms=train_transform,
)
# 이미지/라벨 필터링을 두 번 반복할 필요 없이 같은 샘플 목록에만 전처리를 다르게 적용합니다.
train_full_eval = copy.copy(train_full_aug)
train_full_eval.joint_transform = validation_transform
assert train_full_aug.images == train_full_eval.images

split_generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(train_full_aug), generator=split_generator).tolist()
num_train_pool = int(0.80 * len(permutation))
train_pool, val_pool = permutation[:num_train_pool], permutation[num_train_pool:]
train_indices = train_pool[:min(TRAIN_SAMPLES, len(train_pool))]
val_indices = val_pool[:min(VAL_SAMPLES, len(val_pool))]
train_dataset = Subset(train_full_aug, train_indices)
validation_dataset = Subset(train_full_eval, val_indices)
val_image_ids = [Path(train_full_eval.images[i]).stem for i in val_indices]
assert train_indices and val_indices
assert not (set(train_indices) & set(val_indices))
assert len(val_image_ids) == len(set(val_image_ids))

print('Model input size  :', INPUT_SIZE)
print('Training images  :', len(train_dataset))
print('Validation images:', len(validation_dataset))
print('Test images      : reserved for FINAL TEST (official VOC val.txt)')
print('Random seed      :', SEED)

In [ ]:
# 데이터가 YOLO 입력 및 정답 형식에 맞게 구성됐는지 확인합니다.
example_image, example_target = train_dataset[0]
assert example_image.shape == (3, IMAGE_SIZE, IMAGE_SIZE)
assert example_target.shape == (7, 7, 11)
print('Train / Validation:', len(train_dataset), '/', len(validation_dataset))
print('이미지 텐서:', tuple(example_image.shape))
print('정답 텐서 :', tuple(example_target.shape), '| 객체가 들어있는 셀 수:', int(example_target[..., C].sum()))

## 2. 모델 선택 및 직접 구현

**지원 예시:** ResNet18/34/50, MobileNetV3-Small, EfficientNet-B0. `head_channels`도 변경할 수 있습니다. 이 목록은 제한이 아닙니다. **`register_model()`로 직접 만든 임의의 `torch.nn.Module`을 등록**해도 됩니다.

Baseline의 ResNet18+256채널 Head는 1단계와 구조가 동일합니다. 각 후보는 공정한 비교를 위해 `(N,7,7,16)` 텐서를 반환하도록 하며 **Loss/데이터/평가 함수는 공통**으로 사용합니다. YOLOv8/Faster R-CNN처럼 출력 및 학습 인터페이스가 다른 모델은 별도의 변환 어댑터가 있어야 현재 공통 학습 코드에서 작동합니다.


In [ ]:
# [Stage 2] 모델 구조를 자유롭게 선택할 수 있습니다.
# 기본 ResNet18 구조와 초기값은 Stage 1과 동일합니다.
class YOLOv1(nn.Module):
    SUPPORTED_BACKBONES = (
        'resnet18', 'resnet34', 'resnet50',
        'mobilenet_v3_small', 'efficientnet_b0',
    )

    def __init__(self, S=7, B=2, C=6, pretrained_backbone=True,
                 backbone_name='resnet18', head_channels=256):
        super().__init__()
        self.S, self.B, self.C = S, B, C
        name = backbone_name.lower()
        if name == 'resnet18':
            weights = models.ResNet18_Weights.DEFAULT if pretrained_backbone else None
            backbone = models.resnet18(weights=weights)
            self.features = nn.Sequential(*list(backbone.children())[:-2])
            num_channels = 512
        elif name == 'resnet34':
            weights = models.ResNet34_Weights.DEFAULT if pretrained_backbone else None
            backbone = models.resnet34(weights=weights)
            self.features = nn.Sequential(*list(backbone.children())[:-2])
            num_channels = 512
        elif name == 'resnet50':
            weights = models.ResNet50_Weights.DEFAULT if pretrained_backbone else None
            backbone = models.resnet50(weights=weights)
            self.features = nn.Sequential(*list(backbone.children())[:-2])
            num_channels = 2048
        elif name == 'mobilenet_v3_small':
            weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained_backbone else None
            backbone = models.mobilenet_v3_small(weights=weights)
            self.features = backbone.features
            num_channels = 576
        elif name == 'efficientnet_b0':
            weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained_backbone else None
            backbone = models.efficientnet_b0(weights=weights)
            self.features = backbone.features
            num_channels = 1280
        else:
            raise ValueError(f'등록되지 않은 Backbone: {name}')

        self.head = nn.Sequential(
            nn.Conv2d(num_channels, int(head_channels), kernel_size=3, padding=1),
            nn.BatchNorm2d(int(head_channels)),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(int(head_channels), C + B * 5, kernel_size=1),
        )
        for m in self.head.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (self.S, self.S))
        x = self.head(x)
        return x.permute(0, 2, 3, 1).contiguous()

# 아래 레지스트리에는 학생이 직접 만든 어떠한 nn.Module도 추가할 수 있습니다.
# 유일한 공통 인터페이스: forward([N,3,224,224]) -> [N,7,7,16].
# 예측값을 공통 YOLOv1 Loss 및 VOC mAP 평가 코드에 연결하기 위한 제약입니다.
CUSTOM_MODEL_FACTORIES = {}


def register_model(name, factory):
    if not isinstance(name, str) or not name:
        raise ValueError('모델 이름은 비어 있지 않은 문자열이어야 합니다.')
    if name in YOLOv1.SUPPORTED_BACKBONES:
        raise ValueError(f'{name}은 이미 등록된 기본 모델입니다.')
    if not callable(factory):
        raise TypeError('factory는 (cfg, pretrained_backbone)을 받는 함수여야 합니다.')
    CUSTOM_MODEL_FACTORIES[name] = factory


def build_model(cfg, pretrained_backbone=True):
    name = str(cfg.get('model_name', 'resnet18'))
    if name in YOLOv1.SUPPORTED_BACKBONES:
        return YOLOv1(S=S, B=B, C=C, pretrained_backbone=pretrained_backbone,
                      backbone_name=name, head_channels=int(cfg.get('head_channels', 256)))
    if name in CUSTOM_MODEL_FACTORIES:
        model = CUSTOM_MODEL_FACTORIES[name](cfg, pretrained_backbone)
        if not isinstance(model, nn.Module):
            raise TypeError('사용자 모델 factory는 torch.nn.Module을 반환해야 합니다.')
        return model
    raise ValueError(f'등록되지 않은 모델: {name}. 기본 모델: {YOLOv1.SUPPORTED_BACKBONES}. '
                     f'사용자 모델: {list(CUSTOM_MODEL_FACTORIES)}')


class YoloLoss(nn.Module):
    def __init__(self, S=7, B=2, C=6, lambda_coord=5.0, lambda_noobj=0.5):
        super().__init__()
        self.S, self.B, self.C = S, B, C
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj

    def _iou_for_assignment(self, pred_boxes, gt_boxes):
        # x,y는 cell-relative / w,h는 image-relative이므로 IoU 계산 전 좌표계를 통일.
        gy = torch.arange(self.S, device=pred_boxes.device, dtype=pred_boxes.dtype).view(1, self.S, 1, 1)
        gx = torch.arange(self.S, device=pred_boxes.device, dtype=pred_boxes.dtype).view(1, 1, self.S, 1)

        px = (pred_boxes[..., 0] + gx) / self.S
        py = (pred_boxes[..., 1] + gy) / self.S
        pw = pred_boxes[..., 2].clamp(min=0)
        ph = pred_boxes[..., 3].clamp(min=0)

        tx = (gt_boxes[..., 0] + gx) / self.S
        ty = (gt_boxes[..., 1] + gy) / self.S
        tw = gt_boxes[..., 2].clamp(min=0)
        th = gt_boxes[..., 3].clamp(min=0)

        pxyxy = torch.stack((px-pw/2, py-ph/2, px+pw/2, py+ph/2), -1)
        txyxy = torch.stack((tx-tw/2, ty-th/2, tx+tw/2, ty+th/2), -1)
        inter_lt = torch.maximum(pxyxy[..., :2], txyxy[..., :2])
        inter_rb = torch.minimum(pxyxy[..., 2:], txyxy[..., 2:])
        inter_wh = (inter_rb-inter_lt).clamp(min=0)
        inter = inter_wh[..., 0] * inter_wh[..., 1]
        union = pw*ph + tw*th - inter
        return inter / union.clamp(min=1e-6)

    def forward(self, predictions, target):
        # predictions: (N, 7, 7, 16) / target: (N, 7, 7, 11)
        n = predictions.shape[0]
        assert predictions.shape[1:] == (self.S, self.S, self.C + self.B*5)
        assert target.shape[1:] == (self.S, self.S, self.C + 5)
        pred_cls = predictions[..., :self.C]
        pred_box = predictions[..., self.C:].reshape(n, self.S, self.S, self.B, 5)
        pred_xywh, pred_conf = pred_box[..., :4], pred_box[..., 4]
        target_cls = target[..., :self.C]
        object_mask = target[..., self.C]  # (N, S, S)
        target_xywh = target[..., self.C+1:self.C+5].unsqueeze(-2)

        # 담당 예측기: object가 있는 셀에서 IoU가 최대인 예측 Box
        with torch.no_grad():
            ious = self._iou_for_assignment(pred_xywh.float(), target_xywh.float())
            responsible_idx = ious.argmax(dim=-1)
            conf_target = ious.to(dtype=predictions.dtype).detach()
        responsible_mask = F.one_hot(responsible_idx, num_classes=self.B).to(predictions.dtype)
        responsible_mask = responsible_mask * object_mask.unsqueeze(-1)
        box_mask = responsible_mask.unsqueeze(-1)

        loss_class = ((pred_cls - target_cls).square() * object_mask.unsqueeze(-1)).sum()
        loss_xy = ((pred_xywh[..., :2] - target_xywh[..., :2]).square()
                   * box_mask).sum()
        # 음수 폭/높이 예측에도 gradient가 흐르는 안정적인 signed sqrt
        pred_wh = pred_xywh[..., 2:4]
        pred_wh_sqrt = torch.sign(pred_wh) * torch.sqrt(pred_wh.abs() + 1e-6)
        target_wh_sqrt = torch.sqrt(target_xywh[..., 2:4].clamp(min=0) + 1e-6)
        loss_wh = ((pred_wh_sqrt - target_wh_sqrt).square() * box_mask).sum()
        loss_coord = loss_xy + loss_wh
        loss_obj = ((pred_conf - conf_target).square() * responsible_mask).sum()
        # 배경 셀의 2개 Box + 객체 셀에서 담당하지 않는 Box
        loss_noobj = (pred_conf.square() * (1.0 - responsible_mask)).sum()

        return (self.lambda_coord*loss_coord + loss_obj
                + self.lambda_noobj*loss_noobj + loss_class) / n

# 데이터 없이도 점검할 수 있는 Loss 테스트
_dummy_pred = torch.randn(2, 7, 7, 16, requires_grad=True) * 0.1
_dummy_target = torch.zeros(2, 7, 7, 11)
_dummy_target[0, 2, 3, 0] = 1
_dummy_target[0, 2, 3, 6:11] = torch.tensor([1., 0.5, 0.5, 0.2, 0.3])
_dummy_loss = YoloLoss()(_dummy_pred, _dummy_target)
assert torch.isfinite(_dummy_loss)
_dummy_loss.backward()
print('YOLO Loss 테스트 통과:', round(_dummy_loss.item(), 4))
del _dummy_pred, _dummy_target, _dummy_loss


### 모델 설계 실습 구역 — 마음대로 수정 가능

아래의 `StudentCNN`은 직접 작성한 CNN 예시입니다. 블록, Skip Connection, Attention, Head, 채널 수 등을 자유롭게 바꾸고 `register_model('모델이름', factory)`로 등록하세요. 모델 구조를 수정한 뒤에는 해당 셀과 실험 셀을 다시 실행합니다. 다른 외부 모델도 `nn.Module` 형태로 감싸 같은 텐서를 반환하면 등록할 수 있습니다.


In [ ]:
# Stage 2 선택 과제 예시: 완전히 새로운 소형 CNN 구조를 직접 구현합니다.
# 이 코드는 예제이며, 아키텍처 전체를 자유롭게 바꾸고 등록할 수 있습니다.
class StudentCNN(nn.Module):
    def __init__(self, S=7, B=2, C=6, base_channels=32):
        super().__init__()
        self.S = S
        ch = int(base_channels)
        self.features = nn.Sequential(
            nn.Conv2d(3, ch, 3, stride=2, padding=1), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch*2, 3, stride=2, padding=1), nn.BatchNorm2d(ch*2), nn.ReLU(inplace=True),
            nn.Conv2d(ch*2, ch*4, 3, stride=2, padding=1), nn.BatchNorm2d(ch*4), nn.ReLU(inplace=True),
            nn.Conv2d(ch*4, ch*4, 3, stride=2, padding=1), nn.BatchNorm2d(ch*4), nn.ReLU(inplace=True),
        )
        self.head = nn.Conv2d(ch*4, C + B*5, kernel_size=1)

    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (self.S, self.S))
        return self.head(x).permute(0, 2, 3, 1).contiguous()


register_model(
    'student_cnn',
    lambda cfg, pretrained: StudentCNN(S=S, B=B, C=C,
                                      base_channels=int(cfg.get('student_base_channels', 32)))
)
print('기본 모델:', YOLOv1.SUPPORTED_BACKBONES)
print('등록된 사용자 모델:', list(CUSTOM_MODEL_FACTORIES))


## 3. 공통 평가 함수 — VOC 11-point mAP@0.5

Validation은 VOC 공식 `train.txt`에서 분리한 300장만 사용합니다. 최종 Test는 공식 `val.txt` 전체를 따로 보관합니다. mAP 코드와 `confidence=0.01`, `NMS IoU=0.5`, 정답 IoU=0.5 기준은 **두 노트북에서 동일하게 유지**합니다. 평가 추론만 배치로 수행해 속도를 개선했습니다.


In [ ]:
def iou_torch_eval(box1, box2):
    """ Calculate IoU between two boxes in (x1, y1, x2, y2) format """
    b1_x1, b1_y1, b1_x2, b1_y2 = box1
    b2_x1, b2_y1, b2_x2, b2_y2 = box2

    inter_x1 = max(b1_x1, b2_x1)
    inter_y1 = max(b1_y1, b2_y1)
    inter_x2 = min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2)

    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    b1_area = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    b2_area = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)
    union_area = b1_area + b2_area - inter_area + 1e-6

    return inter_area / union_area

def decode_yolo_output(output, S=7, B=2, C=6, conf_threshold=0.2):
    """ Convert (S, S, B*5+C) tensor to a list of [class, conf, x1, y1, x2, y2] """
    output = output.detach().cpu()

    all_boxes = [] # (class_idx, conf, x1, y1, x2, y2)

    for i in range(S): # grid y
        for j in range(S): # grid x
            # Class
            class_probs = output[i, j, :C] # (C,)
            pred_class_idx = torch.argmax(class_probs).item()
            pred_class_prob = class_probs[pred_class_idx].item()

            # Boxes
            boxes = output[i, j, C:].reshape(B, 5) # (B, 5)

            for b in range(B):
                # (x_cell, y_cell, w_img, h_img, conf)
                box_data = boxes[b]
                confidence = box_data[4].item()

                # class prob * box confidence = final confidence
                final_confidence = pred_class_prob * confidence

                if final_confidence > conf_threshold:
                    # (x_cell, y_cell) -> (x_img, y_img)
                    x_cell, y_cell, w, h = box_data[0:4]
                    x_img = (j + x_cell) / S # 0~1
                    y_img = (i + y_cell) / S # 0~1
                    w_img = w # 0~1
                    h_img = h # 0~1

                    # (x, y, w, h) -> (x1, y1, x2, y2)
                    x1 = x_img - (w_img / 2)
                    y1 = y_img - (h_img / 2)
                    x2 = x_img + (w_img / 2)
                    y2 = y_img + (h_img / 2)

                    all_boxes.append([pred_class_idx, float(final_confidence),
                                      float(x1), float(y1), float(x2), float(y2)])

    return all_boxes

def non_max_suppression(boxes, iou_threshold=0.5):
    """ Perform Non-Max Suppression """
    if not boxes:
        return []

    # Sort by confidence in descending order
    boxes.sort(key=lambda x: x[1], reverse=True)

    final_boxes = []
    while boxes:
        chosen_box = boxes.pop(0)
        final_boxes.append(chosen_box)

        # NMS: Remove boxes of the same class with IoU >= threshold
        boxes = [box for box in boxes
                 if box[0] != chosen_box[0] # if class is different
                 or iou_torch_eval(chosen_box[2:], box[2:]) < iou_threshold # or IoU is low
                ]

    return final_boxes

def visualize_bbox(img_pil, pred_boxes, img_size=INPUT_SIZE):
    """ Visualization function """
    # PIL -> CV2
    img_cv = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
    h, w, _ = img_cv.shape

    # Draw Bbox
    for box in pred_boxes:
        class_idx, conf, x1, y1, x2, y2 = box

        # (0~1) coordinates -> (0~W, 0~H)
        x1_px = int(x1 * w)
        y1_px = int(y1 * h)
        x2_px = int(x2 * w)
        y2_px = int(y2 * h)

        # Class color (simply generate color from class index)
        color = (255 * (class_idx % 3), 255 * ((class_idx * 7) % 11) / 10, 255 * ((class_idx * 3) % 7) / 6)

        # Bbox
        cv2.rectangle(img_cv, (x1_px, y1_px), (x2_px, y2_px), color, 2)

        # Label
        label = f"{VOC_CLASSES[class_idx]}: {conf:.2f}"
        cv2.putText(img_cv, label, (x1_px, y1_px - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # CV2 -> RGB (for plt.imshow)
    img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    return img_cv

def visualize(model, test_img_path):
    # Test Image
    test_img_pil = Image.open(test_img_path).convert('RGB')

    transform_viz = transforms.Compose([
        transforms.Resize(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    test_tensor = transform_viz(test_img_pil).to(device).unsqueeze(0)

    with torch.no_grad():
        conv_out = model(test_tensor) # (1, 7, 7, 16)

    # (S, S, 16) -> list of [class, conf, x1, y1, x2, y2] (0~1)
    pred_boxes_rel = decode_yolo_output(conv_out[0], C=6, conf_threshold=0.1)
    pred_boxes_nms = non_max_suppression(pred_boxes_rel, iou_threshold=0.5)

    # Visualize (pass original PIL image and NMS results)
    vis_output = visualize_bbox(test_img_pil, pred_boxes_nms)

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(test_img_pil) # Original Image
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title("YOLOv1 Prediction (6 Classes)")
    plt.imshow(vis_output) # Image with Bbox drawn
    plt.axis('off')

    plt.show()

In [ ]:
# Fixed 6-class VOC 11-point mAP@0.5. Forward pass is batched for speed.
# image_ids = val_image_ids during tuning; official val.txt is FINAL TEST only.
def evaluate_map(model, image_ids, split_name='Validation', return_details=False, eval_batch_size=32):
    if model is None:
        raise ValueError('No model is available for evaluation.')

    NUM_CLASSES = 6
    conf_thresh = 0.01
    iou_thresh_nms = 0.5
    iou_thresh_map = 0.5
    image_ids = list(image_ids)
    assert image_ids, f'{split_name} image list is empty.'

    print(f'Calculating {split_name} mAP@0.5 on {len(image_ids)} images...')
    model.eval()
    eval_device = next(model.parameters()).device
    transform_conv = transforms.Compose([
        transforms.Resize(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    all_pred_boxes = []  # (image_idx, class_idx, conf, x1, y1, x2, y2)
    all_gt_boxes = []    # (image_idx, class_idx, x1, y1, x2, y2)

    # Model inference only is batched. Decoding/NMS and VOC AP matching follow the
    # same metric protocol as the original notebook.
    with torch.inference_mode():
        for start in tqdm(range(0, len(image_ids), eval_batch_size), desc=f'{split_name} mAP'):
            batch_ids = image_ids[start:start + eval_batch_size]
            batch_imgs, batch_sizes = [], []
            for img_name in batch_ids:
                img_path = VOC_DIR / 'JPEGImages' / f'{img_name}.jpg'
                with Image.open(img_path) as im:
                    img = im.convert('RGB')
                batch_sizes.append(img.size)
                batch_imgs.append(transform_conv(img))

            outputs = model(torch.stack(batch_imgs, dim=0).to(eval_device)).detach().cpu()
            del batch_imgs
            for offset, img_name in enumerate(batch_ids):
                idx = start + offset
                img_w, img_h = batch_sizes[offset]
                boxes = decode_yolo_output(outputs[offset], S=7, B=2, C=6,
                                           conf_threshold=conf_thresh)
                boxes = non_max_suppression(boxes, iou_threshold=iou_thresh_nms)
                for class_idx, conf, x1, y1, x2, y2 in boxes:
                    all_pred_boxes.append([
                        idx, class_idx, conf,
                        x1 * img_w, y1 * img_h, x2 * img_w, y2 * img_h
                    ])

                annot_path = VOC_DIR / 'Annotations' / f'{img_name}.xml'
                tree = ET.parse(annot_path)
                for obj in tree.findall('object'):
                    class_name = obj.find('name').text
                    if class_name not in CLASS_TO_IDX:
                        continue
                    class_idx = CLASS_TO_IDX[class_name]
                    bbox = obj.find('bndbox')
                    x1 = float(bbox.find('xmin').text)
                    y1 = float(bbox.find('ymin').text)
                    x2 = float(bbox.find('xmax').text)
                    y2 = float(bbox.find('ymax').text)
                    all_gt_boxes.append([idx, class_idx, x1, y1, x2, y2])

    average_precisions = []
    ap_by_class = {}
    for c_idx in range(NUM_CLASSES):
        preds_c = [p for p in all_pred_boxes if p[1] == c_idx]
        gts_c = [g for g in all_gt_boxes if g[1] == c_idx]
        if not gts_c:
            # This is the same exclusion rule as in the provided mAP evaluation code.
            if preds_c:
                average_precisions.append(0.0)
                ap_by_class[VOC_CLASSES[c_idx]] = 0.0
            continue

        gt_counter = {}
        gt_detected = {}
        for g in gts_c:
            img_idx = g[0]
            gt_counter.setdefault(img_idx, []).append(g[2:])
            gt_detected.setdefault(img_idx, []).append(False)

        preds_c.sort(key=lambda x: x[2], reverse=True)
        TP = np.zeros(len(preds_c))
        FP = np.zeros(len(preds_c))
        for p_idx, pred in enumerate(preds_c):
            img_idx = pred[0]
            pred_box = pred[3:]
            if img_idx not in gt_counter:
                FP[p_idx] = 1
                continue
            best_iou, best_gt_idx = 0, -1
            for gt_idx, gt_box in enumerate(gt_counter[img_idx]):
                iou = iou_torch_eval(pred_box, gt_box)
                if iou > best_iou:
                    best_iou, best_gt_idx = iou, gt_idx
            if best_iou >= iou_thresh_map:
                if not gt_detected[img_idx][best_gt_idx]:
                    TP[p_idx] = 1
                    gt_detected[img_idx][best_gt_idx] = True
                else:
                    FP[p_idx] = 1
            else:
                FP[p_idx] = 1

        TP_cumsum = np.cumsum(TP)
        FP_cumsum = np.cumsum(FP)
        recalls = TP_cumsum / len(gts_c)
        precisions = TP_cumsum / np.maximum(TP_cumsum + FP_cumsum, 1e-12)

        ap = 0.0
        for t in np.arange(0., 1.1, 0.1):
            precision = np.max(precisions[recalls >= t]) if np.any(recalls >= t) else 0.0
            ap += precision / 11.0

        average_precisions.append(ap)
        ap_by_class[VOC_CLASSES[c_idx]] = float(ap)
        print(f"{split_name} — {VOC_CLASSES[c_idx]} AP: {ap:.4f}")

    mAP = float(np.mean(average_precisions)) if average_precisions else 0.0
    print(f"{split_name} mAP@0.5 (VOC 11-point) = {mAP:.4f}")
    if return_details:
        return mAP, ap_by_class
    return mAP

## 4. 공통 Baseline 학습 (RTX 5080)

공통 시작점: **ResNet18 + AdamW + 224×224 + Train 1,200 / Val 300 + 4 Epoch + batch 16**. GPU 메모리/속도에 따른 수치는 실제 환경에서 달라집니다. **1단계와 2단계의 아래 `BASE_CONFIG` 내용은 완전히 동일**합니다.

**학습 전략:** `pretrained_backbone`, `freeze_backbone`, Optimizer, Scheduler, Warmup, Learning Rate, Weight Decay, Augmentation, Loss Weight, Gradient Accumulation을 실험할 수 있습니다.


In [ ]:
STAGE = 2

# 두 노트북에서 동일한 Baseline. RTX 5080 16GB에서 시작하도록 설계되었습니다.
BASE_CONFIG = {
    'model_name': 'resnet18',             # Stage 1: 고정 / Stage 2: 자유 변경
    'pretrained_backbone': True,         # False: 사전학습 없이 같은 구조를 처음부터 학습
    'epochs': 4,
    'batch_size': 16,
    'optimizer': 'AdamW',                # AdamW / Adam / SGD / RMSprop
    'learning_rate': 3e-4,
    'momentum': 0.9,                     # SGD / RMSprop에 적용
    'weight_decay': 1e-4,
    'scheduler': 'cosine',               # poly / cosine / constant
    'warmup_iters': 10,
    'warmup_factor': 0.1,
    'power': 0.9,
    'scale_range_aug': (0.85, 1.15),
    'color_jitter': (0.2, 0.2, 0.2, 0.05),
    'freeze_backbone': False,           # True: features만 고정하고 head만 학습
    'lambda_coord': 5.0,                # YOLO Loss의 위치 손실 가중치
    'lambda_noobj': 0.5,                # YOLO Loss의 배경 손실 가중치
    'grad_accum_steps': 1,              # 2 이상이면 Gradient Accumulation
    'grad_clip_norm': 1.0,
    'amp': True,
    'num_workers': 2,                   # Windows / notebook 문제 시 0
    'val_every': 2,                     # 2 epoch마다 + 마지막 epoch 평가
    'eval_batch_size': 32,
}

CHECKPOINT_DIR = Path('./checkpoints_stage2')
CHECKPOINT_DIR.mkdir(exist_ok=True, parents=True)
print('현재 단계:', STAGE)
print('Baseline 설정:', BASE_CONFIG)


In [ ]:
# 모델을 제외한 훈련·스케줄러·평가 프로토콜은 두 단계에서 동일합니다.
def create_optimizer(model, cfg):
    name = str(cfg['optimizer']).lower()
    params = [p for p in model.parameters() if p.requires_grad]
    if not params:
        raise ValueError('학습 가능한 파라미터가 없습니다. freeze_backbone / model을 확인하세요.')
    lr, wd = float(cfg['learning_rate']), float(cfg['weight_decay'])
    if name == 'sgd':
        return optim.SGD(params, lr=lr, momentum=float(cfg['momentum']), weight_decay=wd)
    if name == 'adamw':
        return optim.AdamW(params, lr=lr, weight_decay=wd)
    if name == 'adam':
        return optim.Adam(params, lr=lr, weight_decay=wd)
    if name == 'rmsprop':
        return optim.RMSprop(params, lr=lr, momentum=float(cfg['momentum']), weight_decay=wd)
    raise ValueError('optimizer는 SGD / AdamW / Adam / RMSprop 중 하나여야 합니다.')


def create_scheduler(optimizer, cfg, max_iter):
    warmup_iters = min(int(cfg['warmup_iters']), max(0, max_iter - 1))

    def lr_factor(step):
        if warmup_iters and step < warmup_iters:
            return cfg['warmup_factor'] + (1 - cfg['warmup_factor']) * (step / warmup_iters)
        progress = min(1.0, (step - warmup_iters) / max(1, max_iter - warmup_iters))
        name = str(cfg['scheduler']).lower()
        if name == 'poly':
            return max(0.0, 1.0 - progress)**float(cfg['power'])
        if name == 'cosine':
            return 0.5 * (1.0 + np.cos(np.pi * progress))
        if name == 'constant':
            return 1.0
        raise ValueError('scheduler는 poly / cosine / constant 중 하나여야 합니다.')
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_factor)


def train_experiment(name, cfg, init_checkpoint=None, prior_epochs=0):
    if globals().get('_final_test_evaluated', False):
        raise RuntimeError('최종 Test 이후에는 설정을 다시 조정하지 마세요 (Test leakage 방지).')
    cfg = copy.deepcopy(cfg)
    if int(cfg['epochs']) < 1 or int(cfg['batch_size']) < 1:
        raise ValueError('epochs와 batch_size는 1 이상이어야 합니다.')
    if int(cfg['val_every']) < 1 or int(cfg['eval_batch_size']) < 1:
        raise ValueError('val_every와 eval_batch_size는 1 이상이어야 합니다.')
    accum = int(cfg['grad_accum_steps'])
    if accum < 1:
        raise ValueError('grad_accum_steps는 1 이상이어야 합니다.')

    reset_seed(SEED)
    train_transform.scale_range_aug = tuple(cfg['scale_range_aug'])
    train_transform.color_jitter = transforms.ColorJitter(*cfg['color_jitter'])
    loader = DataLoader(
        train_dataset, batch_size=int(cfg['batch_size']), shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
        num_workers=int(cfg['num_workers']), pin_memory=(device.type == 'cuda'),
        persistent_workers=(int(cfg['num_workers']) > 0), drop_last=False,
    )
    model = build_model(cfg, pretrained_backbone=bool(cfg['pretrained_backbone']) and init_checkpoint is None).to(device)
    if init_checkpoint is not None:
        # 이어학습은 같은 구조 / 동일 모델 가중치에서만 가능합니다.
        state = torch.load(init_checkpoint, map_location='cpu', weights_only=True)
        model.load_state_dict(state, strict=True)
        del state
    if cfg.get('freeze_backbone', False):
        feature_extractor = getattr(model, 'features', None)
        if not isinstance(feature_extractor, nn.Module):
            raise ValueError('freeze_backbone=True를 사용하려면 모델이 .features 모듈을 제공해야 합니다.')
        for p in feature_extractor.parameters():
            p.requires_grad_(False)

    # 사용자 정의 모델도 반드시 고정된 YOLO-style 출력 형식을 만족해야 합니다.
    model.eval()
    with torch.inference_mode():
        shape = tuple(model(torch.zeros((1, 3, IMAGE_SIZE, IMAGE_SIZE), device=device)).shape)
    expected_shape = (1, S, S, C + B * 5)
    if shape != expected_shape:
        raise ValueError(f'모델 출력 형식 오류: {shape}, 필요: {expected_shape}. '
                         '사용자 모델의 forward()를 수정해 주세요.')

    parameter_count = sum(p.numel() for p in model.parameters())
    trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print('  Total / Trainable parameters:', f'{parameter_count:,}', '/', f'{trainable_count:,}')
    criterion = YoloLoss(S=S, B=B, C=C,
                         lambda_coord=float(cfg['lambda_coord']),
                         lambda_noobj=float(cfg['lambda_noobj']))
    optimizer = create_optimizer(model, cfg)
    steps_per_epoch = (len(loader) + accum - 1) // accum
    scheduler = create_scheduler(optimizer, cfg, int(cfg['epochs']) * steps_per_epoch)
    use_amp = bool(cfg['amp']) and device.type == 'cuda'
    amp_dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if use_amp else torch.bfloat16
    scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and amp_dtype == torch.float16))

    best_map, best_epoch, best_aps = -1.0, -1, {}
    history = []
    safe_name = ''.join(ch if ch.isalnum() or ch in '-_.' else '_' for ch in name)
    ckpt_path = CHECKPOINT_DIR / f'{safe_name}_best.pth'
    start_time = time.perf_counter()
    start_desc = ('Checkpoint 이어학습' if init_checkpoint is not None else
                  ('사전학습 Backbone' if cfg['pretrained_backbone'] else 'Random / Custom 초기화'))
    print(f'\n▶ {name} | Model={cfg["model_name"]} | 시작={start_desc}')
    for epoch in range(1, int(cfg['epochs']) + 1):
        model.train()
        if cfg.get('freeze_backbone', False):
            model.features.eval()   # BatchNorm 통계도 고정
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        for batch_idx, (images, targets) in enumerate(tqdm(loader, desc=f'{name} {epoch}/{cfg["epochs"]}', leave=False)):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            # 마지막 누적 그룹의 실제 배치 수로 나누어 과도한 축소를 방지합니다.
            remaining = len(loader) - (batch_idx // accum) * accum
            group_size = min(accum, remaining)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                output = model(images)
                raw_loss = criterion(output, targets)
                loss = raw_loss / group_size
            if not torch.isfinite(raw_loss):
                raise FloatingPointError(f'{name}: Loss가 NaN / Inf입니다. LR 등을 조정하세요.')
            scaler.scale(loss).backward()
            if (batch_idx + 1) % accum == 0 or (batch_idx + 1) == len(loader):
                scaler.unscale_(optimizer)
                if float(cfg['grad_clip_norm']) > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(cfg['grad_clip_norm']))
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            running_loss += float(raw_loss.item())

        mean_loss = running_loss / len(loader)
        do_eval = epoch % int(cfg['val_every']) == 0 or epoch == int(cfg['epochs'])
        val_map = None
        if do_eval:
            val_map, ap_by_class = evaluate_map(
                model, val_image_ids, split_name=f'{name} Val epoch {epoch}',
                return_details=True, eval_batch_size=int(cfg['eval_batch_size']))
            if val_map > best_map:
                best_map, best_epoch, best_aps = val_map, epoch, ap_by_class.copy()
                torch.save(model.state_dict(), ckpt_path)
                print(f'  ✓ 최고 Validation 모델 저장: {ckpt_path}')
        history.append({'epoch': epoch, 'train_loss': mean_loss, 'val_map': val_map})
        val_msg = f'{val_map:.4f}' if val_map is not None else '(이번 Epoch 생략)'
        print(f'[{name}] Epoch {epoch:02d}/{cfg["epochs"]} | Loss={mean_loss:.3f} | Val mAP={val_msg}')

    elapsed_seconds = time.perf_counter() - start_time
    print(f'  학습+검증: {elapsed_seconds:.1f}초 (GPU/SSD/환경에 따라 달라집니다)')
    del loader, optimizer, scheduler, scaler, criterion, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        'name': name, 'checkpoint': str(ckpt_path), 'best_val_map': float(best_map),
        'best_epoch': best_epoch, 'best_aps': best_aps, 'history': history,
        'config': copy.deepcopy(cfg), 'init_checkpoint': str(init_checkpoint) if init_checkpoint else None,
        'prior_epochs': int(prior_epochs), 'total_epochs': int(prior_epochs) + int(cfg['epochs']),
        'elapsed_seconds': elapsed_seconds, 'parameters': parameter_count,
        'trainable_parameters': trainable_count, 'initialization': start_desc,
    }


def load_model_checkpoint(path, cfg):
    # architecture, head width, custom model parameters 모두 실험 당시의 config로 재현.
    model = build_model(cfg, pretrained_backbone=False)
    state = torch.load(path, map_location='cpu', weights_only=True)
    model.load_state_dict(state, strict=True)
    del state
    return model.to(device).eval()


def plot_validation_history(run):
    h = pd.DataFrame(run['history'])
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    axes[0].plot(h['epoch'], h['train_loss'], marker='o')
    axes[0].set(title='Train Loss', xlabel='Epoch', ylabel='Loss')
    axes[1].plot(h['epoch'], h['val_map'], marker='o')
    axes[1].set(title='Validation mAP@0.5', xlabel='Epoch', ylabel='mAP')
    axes[1].set_ylim(bottom=0, top=max(0.05, min(1.0, 1.15 * h['val_map'].max())))
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.xaxis.get_major_locator().set_params(integer=True)
    fig.suptitle(run['name'])
    fig.tight_layout()
    plt.show()


def plot_ap(aps, title):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    values = [aps.get(cls, 0.) for cls in VOC_CLASSES]
    ax.bar(VOC_CLASSES, values)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('AP@0.5 (VOC 11-point)')
    ax.set_title(title)
    for i, val in enumerate(values):
        ax.text(i, min(float(val) + 0.02, 0.98), f'{val:.3f}', ha='center', fontsize=9)
    fig.tight_layout()
    plt.show()


In [ ]:
# Baseline을 먼저 완주하세요. 두 단계 모두 ResNet18 + 동일 설정입니다.
results = []
baseline = train_experiment('baseline', BASE_CONFIG)
results.append(baseline)
print('\n★ Baseline 최고 Validation mAP@0.5:', f'{baseline["best_val_map"]:.4f}')
plot_validation_history(baseline)
plot_ap(baseline['best_aps'], 'Baseline — Validation AP by Class')


## 5. 정성 평가: 실제 이미지의 정답 / 예측 Box 비교

Validation 이미지 3장의 원본/정답/예측 Bounding Box를 비교합니다. 그림만 보기 위한 confidence threshold는 0.1로 설정하며, 정량 평가의 0.01과는 다릅니다.


In [ ]:
# 정성 평가에는 Validation 이미지만 사용합니다 (최종 Test 이미지 접근 금지).
def qualitative_compare(model, image_id, viz_threshold=0.1, max_boxes=15):
    jpg = VOC_DIR / 'JPEGImages' / f'{image_id}.jpg'
    xml = VOC_DIR / 'Annotations' / f'{image_id}.xml'
    image = Image.open(jpg).convert('RGB')
    original = np.asarray(image).copy()
    gt_vis = original.copy()
    pred_vis = original.copy()
    h, w = original.shape[:2]

    root = ET.parse(xml).getroot()
    for obj in root.findall('object'):
        label = obj.find('name').text
        if label not in CLASS_TO_IDX:
            continue
        box = obj.find('bndbox')
        x1, y1, x2, y2 = [int(float(box.find(k).text)) for k in ('xmin','ymin','xmax','ymax')]
        cv2.rectangle(gt_vis, (x1,y1), (x2,y2), (0,185,35), 2)
        cv2.putText(gt_vis, label, (x1,max(12,y1-5)), cv2.FONT_HERSHEY_SIMPLEX,
                    .5, (0,185,35), 2)

    inp = transforms.Compose([
        transforms.Resize(INPUT_SIZE), transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(inp)[0]
    boxes = decode_yolo_output(output, S=7, B=2, C=6, conf_threshold=viz_threshold)
    boxes = non_max_suppression(boxes, iou_threshold=0.5)[:max_boxes]
    for cls, confidence, x1, y1, x2, y2 in boxes:
        a = (int(x1*w),int(y1*h))
        b = (int(x2*w),int(y2*h))
        cv2.rectangle(pred_vis, a, b, (220,40,40), 2)
        cv2.putText(pred_vis, f'{VOC_CLASSES[cls]} {confidence:.2f}',
                    (a[0],max(12,a[1]-5)), cv2.FONT_HERSHEY_SIMPLEX, .5, (220,40,40), 2)

    fig, axes = plt.subplots(1,3,figsize=(16,4))
    for ax, img, title in zip(axes, [original,gt_vis,pred_vis],
                              ['Original', 'Ground Truth', 'YOLOv1 Prediction']):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'Validation image: {image_id} | Visualization threshold: {viz_threshold}')
    fig.tight_layout()
    plt.show()

baseline_model = load_model_checkpoint(baseline['checkpoint'], cfg=baseline['config'])
for image_id in val_image_ids[:3]:
    qualitative_compare(baseline_model, image_id)
del baseline_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 6. 학생 실험: **모델과 학습 방법 모두 변경**

모델별 실험을 `SELECTED_EXPERIMENTS`에 추가하고 `RUN_EXTRA_EXPERIMENTS=True`로 변경합니다. `student_cnn`은 직접 작성 모델이고, 그 외 모델은 torchvision의 ImageNet 사전학습 Backbone을 사용합니다. 다른 모델/Head는 `EXPERIMENT_LIBRARY`와 `register_model()`을 확장하면 됩니다.

**공정 비교:** Backbone 변경 실험은 같은 1,200장/300장과 동일 해상도, 동일 초기 학습 Epoch 수를 사용합니다. 파라미터 수/학습 시간도 함께 기록합니다. Baseline에서 이어서 학습한 실험은 총 Epoch 수가 더 많으므로 별도로 해석합니다.


In [ ]:
RUN_EXTRA_EXPERIMENTS = False
SELECTED_EXPERIMENTS = ['resnet34', 'mobilenet_v3_small', 'student_cnn']

# Stage 2: 데이터/평가 기준은 같고, 모델/학습 전략을 모두 자유롭게 변경합니다.
EXPERIMENT_LIBRARY = {
    'resnet34':           {'mode': 'fresh', 'params': {'model_name': 'resnet34'}},
    'resnet50':           {'mode': 'fresh', 'params': {'model_name': 'resnet50'}},
    'mobilenet_v3_small': {'mode': 'fresh', 'params': {'model_name': 'mobilenet_v3_small'}},
    'efficientnet_b0':    {'mode': 'fresh', 'params': {'model_name': 'efficientnet_b0'}},
    'narrow_head':        {'mode': 'fresh', 'params': {'head_channels': 128}},
    'student_cnn':        {'mode': 'fresh', 'params': {'model_name': 'student_cnn', 'pretrained_backbone': False}},
    'sgd':                {'mode': 'fresh', 'params': {'optimizer': 'SGD', 'learning_rate': 1e-2}},
    'lower_lr':           {'mode': 'fresh', 'params': {'learning_rate': 1e-4}},
    'weaker_aug':         {'mode': 'fresh', 'params': {'scale_range_aug': (0.95, 1.05)}},
    'freeze_backbone':    {'mode': 'fresh', 'params': {'freeze_backbone': True, 'learning_rate': 1e-3}},
    'fine_tune_low_lr':   {'mode': 'continue', 'params': {'epochs': 2, 'learning_rate': 1e-4}},
}


if RUN_EXTRA_EXPERIMENTS:
    for trial_name in SELECTED_EXPERIMENTS:
        if trial_name not in EXPERIMENT_LIBRARY:
            raise KeyError(f'정의되지 않은 실험: {trial_name}. EXPERIMENT_LIBRARY에 직접 추가하세요.')
        spec = EXPERIMENT_LIBRARY[trial_name]
        new_cfg = {**BASE_CONFIG, **spec['params']}
        run_name, version = trial_name, 2
        existing = {run['name'] for run in results}
        while run_name in existing:
            run_name = f'{trial_name}_v{version}'
            version += 1
        init_path = baseline['checkpoint'] if spec['mode'] == 'continue' else None
        extra_epochs = baseline['total_epochs'] if init_path else 0
        if init_path and new_cfg['model_name'] != baseline['config']['model_name']:
            raise ValueError('이어서 학습하려면 Baseline과 동일한 모델 구조여야 합니다.')
        trial = train_experiment(run_name, new_cfg, init_checkpoint=init_path,
                                 prior_epochs=extra_epochs)
        results.append(trial)
        plot_validation_history(trial)
else:
    print('Baseline까지 완료. 실습 실험을 실행하려면 RUN_EXTRA_EXPERIMENTS=True로 변경하세요.')


### 원하는 조합을 직접 추가해 보기

자동 실험 목록에 없는 설정은 아래 `MY_OVERRIDES`에 지정합니다. 새 모델/전략은 `fresh`로, 같은 모델을 추가 Fine-tuning할 때만 `continue`를 사용하세요. 결과는 `results`에 누적됩니다. 여러 번 실행하면 실험 이름에 버전이 자동으로 붙습니다.


In [ ]:
# 자유 실험 추가 셀: 기존 자동 실험 목록에 없는 조합을 시도할 때 사용합니다.
RUN_MY_EXPERIMENT = False
MY_EXPERIMENT_NAME = 'my_experiment_01'
MY_OVERRIDES = {
    'learning_rate': 2e-4,
    'optimizer': 'AdamW',
    # Stage 1에서는 model_name/head_channels를 바꾸지 마세요.
    # Stage 2에서는 예: 'model_name': 'student_cnn', 'student_base_channels': 48
}
MY_MODE = 'fresh'   # fresh: 새 학습, continue: Baseline 최고 가중치에서 이어 학습

if RUN_MY_EXPERIMENT:
    cfg = {**BASE_CONFIG, **MY_OVERRIDES}
    if MY_MODE not in {'fresh', 'continue'}:
        raise ValueError('MY_MODE는 fresh 또는 continue여야 합니다.')
    init_checkpoint = baseline['checkpoint'] if MY_MODE == 'continue' else None
    if init_checkpoint and (cfg['model_name'] != baseline['config']['model_name'] or
                           cfg.get('head_channels', 256) != baseline['config'].get('head_channels', 256)):
        raise ValueError('다른 모델 구조로 변경하려면 fresh부터 시작하세요.')
    my_name, suffix = MY_EXPERIMENT_NAME, 2
    while my_name in {r['name'] for r in results}:
        my_name = f'{MY_EXPERIMENT_NAME}_v{suffix}'
        suffix += 1
    trial = train_experiment(my_name, cfg,
                             init_checkpoint=init_checkpoint,
                             prior_epochs=(baseline['total_epochs'] if init_checkpoint else 0))
    results.append(trial)
    plot_validation_history(trial)


## 7. 결과 비교 및 최종 모델 선택

**Validation mAP@0.5**로 최종 모델을 선택합니다. 파라미터 수, 학습 시간, 이번 학습 Epoch와 누적 학습 Epoch를 함께 확인하세요. 동일 Epoch 실험과 추가 Fine-tuning을 구분하여 해석해야 합니다. 비교 결과는 `experiments_stage*.csv`에도 기록합니다.


In [ ]:
comparison = pd.DataFrame([{
    '실험': r['name'],
    '모델': r['config']['model_name'],
    '초기화': r['initialization'],
    '최고 Val mAP@0.5': r['best_val_map'],
    'Best Epoch': r['best_epoch'],
    '이번 학습 Epoch': r['config']['epochs'],
    '누적 학습 Epoch': r['total_epochs'],
    'Learning Rate': r['config']['learning_rate'],
    'Optimizer': r['config']['optimizer'],
    'Augmentation': str(r['config']['scale_range_aug']),
    'Freeze': r['config']['freeze_backbone'],
    '전체 파라미터': r['parameters'],
    '학습 파라미터': r['trainable_parameters'],
    '소요 시간(초)': round(r['elapsed_seconds'], 1),
} for r in results]).sort_values('최고 Val mAP@0.5', ascending=False).reset_index(drop=True)
display(comparison)

fig, ax = plt.subplots(figsize=(max(7, 1.1*len(comparison)), 3.5))
ax.bar(comparison['실험'], comparison['최고 Val mAP@0.5'])
ax.set(title='Validation mAP@0.5 — Experiment Comparison', ylabel='mAP@0.5', ylim=(0, 1.0))
ax.tick_params(axis='x', labelrotation=15)
fig.tight_layout()
plt.show()

best_run = max(results, key=lambda r: r['best_val_map'])
print('선택된 최종 모델 (Validation 기준):', best_run['name'])
print('모델      :', best_run['config']['model_name'])
print('Validation mAP@0.5:', f'{best_run["best_val_map"]:.4f}')
print('Checkpoint:', best_run['checkpoint'])

# 최종 모델을 고르는 데 Test 수치는 사용하지 않습니다.
comparison.to_csv(f'experiments_stage{STAGE}.csv', index=False, encoding='utf-8-sig')


## 8. Final Test — 모델과 설정을 모두 확정한 다음 **딱 한 번**

공식 VOC2012 `val.txt` 전체에서 mAP@0.5를 계산합니다. `RUN_FINAL_TEST=True`로 바꿔 아래 셀을 실행하면 Stage별 최종 가중치도 저장합니다. Test 결과를 본 뒤 추가 튜닝 및 재평가를 하지 않습니다.


In [ ]:
RUN_FINAL_TEST = False  # 모든 실험 및 모델 선택 이후 True로 바꾸세요. 1회만 평가합니다.

if not RUN_FINAL_TEST:
    print('Final Test는 수행하지 않았습니다. Validation 기반 튜닝 후 한 번만 실행하세요.')
else:
    if globals().get('_final_test_evaluated', False):
        raise RuntimeError('이 런타임에서는 이미 최종 Test를 완료했습니다.')
    if best_run is None or not Path(best_run['checkpoint']).is_file():
        raise RuntimeError('먼저 Validation에서 가장 좋은 모델을 선택해 주세요.')

    # 공식 VOC2012 val.txt는 최종 테스트 전용 (Train/Val은 VOC train.txt만 사용).
    with open(VOC_DIR / 'ImageSets' / 'Main' / 'val.txt', 'r', encoding='utf-8') as f:
        final_test_ids = [line.strip() for line in f if line.strip()]
    assert not (set(final_test_ids) & set(val_image_ids)), 'Validation/Test가 겹칩니다.'
    assert not (set(final_test_ids) & {Path(train_full_aug.images[i]).stem for i in train_indices}), 'Train/Test가 겹칩니다.'

    model_path = Path(f'YOLO_stage{STAGE}_best.pth')
    shutil.copyfile(best_run['checkpoint'], model_path)
    final_model = load_model_checkpoint(model_path, cfg=best_run['config'])
    final_test_map, test_aps = evaluate_map(
        final_model, final_test_ids, split_name='Final Test',
        return_details=True, eval_batch_size=int(best_run['config']['eval_batch_size']))
    _final_test_evaluated = True
    print('\n' + '='*65)
    print('선택된 실험     :', best_run['name'])
    print('모델           :', best_run['config']['model_name'])
    print('Best Val mAP   :', f'{best_run["best_val_map"]:.4f}')
    print('Final Test mAP :', f'{final_test_map:.4f}')
    print('모델 파일       :', model_path.resolve())
    for cls in VOC_CLASSES:
        print(f'  {cls:>8}: {test_aps.get(cls, float("nan")):.4f}')
    print('='*65)
    plot_ap(test_aps, 'Final Test — AP by Class')
    del final_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
